# Algorithm 6 — Token & Cache Economics

**File:** `src/CopilotScope.Collector/Quality/Analyzers.cs` (lines 134–211)

Estimates session cost from a configurable per-model price table and quantifies prompt-cache
savings.  Results are **relative estimates**, not billing statements.

---

## 1  Pricing Model

Prices are in **USD per million tokens ($/MTok)**.

| Model | $p_{\text{in}}$ | $p_{\text{out}}$ | $p_{\text{cache}}$ |
|---|---|---|---|
| `claude-opus` | 15.00 | 75.00 | 1.50 |
| `claude-sonnet` | 3.00 | 15.00 | 0.30 |
| `claude-haiku` | 0.80 | 4.00 | 0.08 |
| `gpt-4o` | 2.50 | 10.00 | 1.25 |
| `gpt-4o-mini` | 0.15 | 0.60 | 0.075 |
| `gpt-4.1` | 2.00 | 8.00 | 0.50 |
| `default` | 3.00 | 15.00 | 0.30 |

**Model resolution:** longest-prefix match (so `gpt-4o-mini` is not consumed by `gpt-4o`).

---

## 2  Cost Formula

For each model $m$ with token counts $(I_m, O_m, C_m)$ = (input, output, cache-read):

$$
\text{Cost}_m = \frac{I_m}{10^6} \cdot p^m_{\text{in}}
              + \frac{O_m}{10^6} \cdot p^m_{\text{out}}
              + \frac{C_m}{10^6} \cdot p^m_{\text{cache}}
$$

$$
\text{Total cost} = \sum_m \text{Cost}_m
$$

---

## 3  Cache Savings

Each cache-read token costs $p_{\text{cache}}$ instead of $p_{\text{in}}$:

$$
\text{Savings}_m = \frac{C_m}{10^6} \cdot \max\!\left(0,\; p^m_{\text{in}} - p^m_{\text{cache}}\right)
$$

---

## 4  Cache Hit Ratio (score)

$$
\text{Score} = r_{\text{cache}} = \frac{\sum_m C_m}{\sum_m (I_m + C_m)}
$$

A ratio $\geq 0.5$ indicates heavy cache reuse; $< 0.1$ with large context flags fresh-context waste.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

@dataclass
class ModelPrice:
    input: float
    output: float
    cache_read: float

PRICES = {
    'default':       ModelPrice(3.00,  15.00, 0.30),
    'gpt-4o-mini':   ModelPrice(0.15,   0.60, 0.075),
    'gpt-4o':        ModelPrice(2.50,  10.00, 1.25),
    'gpt-4.1':       ModelPrice(2.00,   8.00, 0.50),
    'claude-haiku':  ModelPrice(0.80,   4.00, 0.08),
    'claude-sonnet': ModelPrice(3.00,  15.00, 0.30),
    'claude-opus':   ModelPrice(15.00, 75.00, 1.50),
}

def resolve_price(model: str) -> ModelPrice:
    candidates = [(k, v) for k, v in PRICES.items()
                  if k != 'default' and k.lower() in model.lower()]
    if candidates:
        return max(candidates, key=lambda x: len(x[0]))[1]
    return PRICES['default']

def session_economics(model_usage: dict) -> dict:
    total_cost = total_savings = 0.0
    total_input = total_cache = 0
    for model, u in model_usage.items():
        p = resolve_price(model)
        I, O, C = u['input'], u['output'], u['cache_read']
        cost    = I/1e6 * p.input + O/1e6 * p.output + C/1e6 * p.cache_read
        savings = C/1e6 * max(0, p.input - p.cache_read)
        total_cost    += cost
        total_savings += savings
        total_input   += I
        total_cache   += C
    prompt_tokens = total_input + total_cache
    cache_ratio = total_cache / prompt_tokens if prompt_tokens > 0 else 0
    return {'cost': total_cost, 'savings': total_savings, 'cache_ratio': cache_ratio}

# --- Prefix resolution test ---
print('=== Longest-prefix resolution ===')
for m in ['gpt-4o', 'gpt-4o-mini', 'claude-sonnet-20240620', 'unknown-model']:
    p = resolve_price(m)
    print(f'  {m:<35} -> in=${p.input}, out=${p.output}')

# --- Session cost scenarios ---
print()
sessions = [
    ('Sonnet, heavy cache reuse', {
        'claude-sonnet': {'input': 50_000, 'output': 10_000, 'cache_read': 150_000}}),
    ('GPT-4o, no cache', {
        'gpt-4o': {'input': 80_000, 'output': 20_000, 'cache_read': 0}}),
    ('Multi-model session', {
        'claude-haiku':  {'input': 30_000, 'output': 8_000, 'cache_read': 70_000},
        'claude-sonnet': {'input': 20_000, 'output': 5_000, 'cache_read': 60_000}}),
    ('GPT-4o-mini, budget session', {
        'gpt-4o-mini': {'input': 100_000, 'output': 40_000, 'cache_read': 200_000}}),
]

print(f'{"Scenario":<40} {"Cost ($)":>10} {"Savings ($)":>12} {"Cache ratio":>12}')
print('-' * 78)
for name, usage in sessions:
    r = session_economics(usage)
    print(f'{name:<40} {r["cost"]:>10.4f} {r["savings"]:>12.4f} {r["cache_ratio"]:>12.1%}')

In [ ]:
# Cost per model family, input vs output breakdown
models = ['claude-haiku', 'gpt-4o-mini', 'gpt-4.1', 'gpt-4o', 'claude-sonnet', 'claude-opus']
tok_in  = 50_000
tok_out = 15_000
in_costs  = [resolve_price(m).input  * tok_in  / 1e6 for m in models]
out_costs = [resolve_price(m).output * tok_out / 1e6 for m in models]

x = np.arange(len(models))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, in_costs,  label=f'Input ({tok_in//1000}k tokens)', color='steelblue')
ax.bar(x, out_costs, bottom=in_costs, label=f'Output ({tok_out//1000}k tokens)', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set_ylabel('Estimated cost (USD)')
ax.set_title('Session cost by model — same token volumes')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('token_economics_cost.png', dpi=150)
plt.show()